In [1]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import RegularPolygon
from matplotlib.collections import PatchCollection
import stim

import os 

while 'honeycomb.py' not in os.listdir():
    os.chdir("..")

from honeycomb import CSSHoneycomb
from operations import Detector


from sims import run_experiment, build_circuit, check_circuit,estimate_logical_error_rate
from collections import defaultdict
import json

In [2]:
result = run_experiment(config_filepath="config/5x3.json", max_shots=200_000, seed=1)
print(result)

shots: 200000
logical error rate (any logical): 0.06674  [95% CI 0.06565, 0.06784]  (13347 failures)
  Z logical: 0.02968  (5936 failures)
  X logical: 0.04229  (8459 failures)


Post-selection uses the same circuit as the cell above. It samples 200,000 shots with `compile_detector_sampler(seed=1)` and `separate_observables=True`.

- A shot is rejected when any detector is 1, including detectors from the final readout.
- A shot is accepted when every detector is 0.
- An accepted shot is a logical error when its sampled observable is 1. There is no correction. Z is observable 0 and X is observable 1, as in `build_circuit`.


In [5]:
from sims import wilson_interval

c = build_circuit(config_filepath="config/5x3.json")
shots = 200_000
dets, obs = c.compile_detector_sampler(seed=1).sample(shots, separate_observables=True)

accepted = ~dets.any(axis=1)
n_accepted = int(accepted.sum())
n_rejected = shots - n_accepted
acceptance = n_accepted / shots
rejection = n_rejected / shots
acc_lo, acc_hi = wilson_interval(n_accepted, shots)

print(f"shots: {shots}")
print(f"accepted: {n_accepted}  rejected: {n_rejected}")
print(f"acceptance rate: {acceptance:.4g}  [95% CI {acc_lo:.4g}, {acc_hi:.4g}]")
print(f"rejection rate: {rejection:.4g}")

if n_accepted == 0:
    print("no shots accepted")
else:
    kept = obs[accepted]
    labels = ("Z", "X", "either")
    counts = [
        int(kept[:, 0].sum()),
        int(kept[:, 1].sum()),
        int(kept.any(axis=1).sum()),
    ]
    for label, k in zip(labels, counts):
        lo, hi = wilson_interval(k, n_accepted)
        print(
            f"  {label} logical error rate among accepted: {k / n_accepted:.4g}"
            f"  [95% CI {lo:.4g}, {hi:.4g}]  ({k} failures)"
        )


shots: 200000
accepted: 146733  rejected: 53267
acceptance rate: 0.7337  [95% CI 0.7317, 0.7356]
rejection rate: 0.2663
  Z logical error rate among accepted: 0.002222  [95% CI 0.001993, 0.002476]  (326 failures)
  X logical error rate among accepted: 0.007994  [95% CI 0.007551, 0.008463]  (1173 failures)
  either logical error rate among accepted: 0.009269  [95% CI 0.008791, 0.009772]  (1360 failures)


In [3]:
c = build_circuit(config_filepath="config/5x3.json", error_model="error_model.json")
check_circuit(c)

# expected faults per shot
dem = c.detector_error_model(decompose_errors=True, approximate_disjoint_errors=True,
                             ignore_decomposition_failures=True)
print("expected faults/shot:", sum(i.args_copy()[0] for i in dem.flattened() if i.type == "error"))

# smallest set of faults that flips a logical with no detection event
print(len(c.search_for_undetectable_logical_errors(
    dont_explore_detection_event_sets_with_size_above=4,
    dont_explore_edges_with_degree_above=3,
    dont_explore_edges_increasing_symptom_degree=False)))

expected faults/shot: 0.3171944441889735
1


In [4]:
c = build_circuit(config_filepath="config/5x3.json", error_model="error_model.json")

# 1. the shortest one, with its location in the circuit
errs = c.search_for_undetectable_logical_errors(
    dont_explore_detection_event_sets_with_size_above=4,
    dont_explore_edges_with_degree_above=3,
    dont_explore_edges_increasing_symptom_degree=False)
print(errs[0])

# 2. all single faults with no detection events, and their total probability
dem = c.detector_error_model(decompose_errors=True, approximate_disjoint_errors=True, ignore_decomposition_failures=True)
undetected = [i for i in dem.flattened()
              if i.type == "error"
              and not any(t.is_relative_detector_id() for t in i.targets_copy())]
print(len(undetected), "mechanisms, total p =", sum(i.args_copy()[0] for i in undetected))

# 3. where each one comes from
bad = stim.DetectorErrorModel()
for i in undetected:
    bad.append(i)
for e in c.explain_detector_error_model_errors(dem_filter=bad, reduce_to_one_representative_error=True):
    print(e.dem_error_terms, "<-", e.circuit_error_locations[0])

ExplainedError {
    dem_error_terms: L1
    CircuitErrorLocation {
        flipped_pauli_product: Y20[coords 2,0.5]*Z8[coords 2,1]
        Circuit location stack trace:
            (after 9 TICKs)
            at instruction #82 (DEPOLARIZE2) in the circuit
            at targets #3 to #4 of the instruction
            resolving to DEPOLARIZE2(0.001) 20[coords 2,0.5] 8[coords 2,1]
    }
    CircuitErrorLocation {
        flipped_pauli_product: Z20[coords 2,0.5]*Z8[coords 2,1]
        Circuit location stack trace:
            (after 9 TICKs)
            at instruction #82 (DEPOLARIZE2) in the circuit
            at targets #3 to #4 of the instruction
            resolving to DEPOLARIZE2(0.001) 20[coords 2,0.5] 8[coords 2,1]
    }
    CircuitErrorLocation {
        flipped_pauli_product: Z8[coords 2,1]
        Circuit location stack trace:
            (after 8 TICKs)
            at instruction #79 (DEPOLARIZE1) in the circuit
            at target #4 of the instruction
            resolv

In [5]:
base = json.load(open("error_model.json"))
for scale in (1, 0.5, 0.25):
    m = {k: (v * scale if k.startswith("p_") else v) for k, v in base.items()}
    json.dump(m, open("tmp_model.json", "w"))
    r = run_experiment(config_filepath="config/5x3.json", error_model="tmp_model.json", max_shots=200_000, seed=1)
    print(scale, r.logical_error_rate)

1 0.06653
0.5 0.03323
0.25 0.016695


In [4]:
c = build_circuit(config_filepath="config/5x3.json", error_model="error_model.json")
dem = c.detector_error_model(decompose_errors=True, approximate_disjoint_errors=True, ignore_decomposition_failures=True)

clean = stim.DetectorErrorModel()
for i in dem.flattened():
    if i.type == "error" and not any(t.is_relative_detector_id() for t in i.targets_copy()):
        continue                      # skip the undetectable injection faults
    clean.append(i)

path = clean.shortest_graphlike_error()
print("effective distance:", len(path))
for e in c.explain_detector_error_model_errors(dem_filter=path, reduce_to_one_representative_error=True):
    print(e.dem_error_terms, "<-", e.circuit_error_locations[0])

effective distance: 2
[stim.DemTargetWithCoords(dem_target=stim.DemTarget('D0'), coords=[1.75, 0, 5]), stim.DemTargetWithCoords(dem_target=stim.DemTarget('D1'), coords=[2.5, 0, 5])] <- CircuitErrorLocation {
    flipped_pauli_product: X4[coords 3,0]
    Circuit location stack trace:
        (after 0 TICKs)
        at instruction #39 (X_ERROR) in the circuit
        at target #4 of the instruction
        resolving to X_ERROR(0.001) 4[coords 3,0]
}
[stim.DemTargetWithCoords(dem_target=stim.DemTarget('D0'), coords=[1.75, 0, 5]), stim.DemTargetWithCoords(dem_target=stim.DemTarget('D1'), coords=[2.5, 0, 5]), stim.DemTargetWithCoords(dem_target=stim.DemTarget('L0'), coords=[])] <- CircuitErrorLocation {
    flipped_pauli_product: X3[coords 2,0]
    Circuit location stack trace:
        (after 0 TICKs)
        at instruction #39 (X_ERROR) in the circuit
        at target #3 of the instruction
        resolving to X_ERROR(0.001) 3[coords 2,0]
}


In [7]:
dem = c.detector_error_model(approximate_disjoint_errors=True)   # no decomposition: ideal decoder

groups = defaultdict(lambda: defaultdict(float))   # syndrome -> {observable set -> total p}
for inst in dem.flattened():
    if inst.type != "error":
        continue
    p = inst.args_copy()[0]
    ts = inst.targets_copy()
    dets = frozenset(t.val for t in ts if t.is_relative_detector_id())
    obs = frozenset(t.val for t in ts if t.is_logical_observable_id())
    groups[dets][obs] += p

predicted = 0.0
for dets, classes in groups.items():
    if not dets:                       # undetectable: only the no-flip outcome is benign
        predicted += sum(p for o, p in classes.items() if o)
    else:                              # decoder picks the most likely class; the rest are failures
        predicted += sum(classes.values()) - max(classes.values())
print("first-order prediction of failure rate:", predicted)

first-order prediction of failure rate: 0.058869246044569654


In [8]:
def first_order(c):
    from collections import defaultdict
    dem = c.detector_error_model(approximate_disjoint_errors=True)
    g = defaultdict(lambda: defaultdict(float))
    for i in dem.flattened():
        if i.type != "error": continue
        ts = i.targets_copy()
        d = frozenset(t.val for t in ts if t.is_relative_detector_id())
        o = frozenset(t.val for t in ts if t.is_logical_observable_id())
        g[d][o] += i.args_copy()[0]
    return sum(
        sum(p for o, p in cl.items() if o) if not d else sum(cl.values()) - max(cl.values())
        for d, cl in g.items()
    )

base = json.load(open("error_model.json"))
for scale in (1, 0.5, 0.25, 0.1):
    m = {k: (v * scale if k.startswith("p_") else v) for k, v in base.items()}
    json.dump(m, open("tmp_model.json", "w"))
    c = build_circuit(config_filepath="config/5x3.json", error_model="tmp_model.json")
    r = estimate_logical_error_rate(c, max_shots=400_000, seed=1)
    pred = first_order(c)
    print(f"scale {scale}: measured {r.logical_error_rate:.4f}  predicted {pred:.4f}  ratio {r.logical_error_rate / pred:.2f}")

scale 1: measured 0.0669  predicted 0.0589  ratio 1.14
scale 0.5: measured 0.0334  predicted 0.0295  ratio 1.13
scale 0.25: measured 0.0169  predicted 0.0148  ratio 1.14
scale 0.1: measured 0.0065  predicted 0.0059  ratio 1.10


In [9]:
from sims import *

c = build_circuit(config_filepath="config/5x3.json", error_model="error_model.json")
m = build_matching(c)

rate, worst = first_order_failure_rate(c, m, top=10)
print("first-order failure rate of this decoder:", rate)      # compare with 0.0589 ideal, 0.1087 measured
for p, s in worst:
    print(f"{p:.2e}  {s}")

mc = build_matching(c, correlated=True)
print("correlated, first-order:", first_order_failure_rate(c, mc, correlated=True)[0])
print(estimate_logical_error_rate(c, mc, max_shots=200_000, seed=1, correlated=True))

first-order failure rate of this decoder: 0.06639543130347583
8.34e-03  D[10] L[0]
7.15e-03  D[15] L[0]
6.69e-03  D[9] L[1]
6.30e-03  D[] L[1]
5.24e-03  D[7] L[]
5.24e-03  D[8] L[1]
4.25e-03  D[13] L[1]
2.26e-03  D[12] L[]
2.20e-03  D[9, 10] L[0]
1.93e-03  D[2] L[0]
correlated, first-order: 0.11623742232085651
shots: 200000
logical error rate (any logical): 0.1077  [95% CI 0.1064, 0.1091]  (21544 failures)
  Z logical: 0.06412  (12823 failures)
  X logical: 0.0499  (9980 failures)
